# F-12 span extractor — GPU training

Trains the data-type and purpose span extractor for the Cross-Source Privacy
Disclosure Consistency Auditor. Runs on a free Colab or Kaggle GPU; the
8 GB laptop is not used.

**This notebook does not reimplement anything.** It unzips the bundle
produced by `python ml/export_f12_bundle.py` and runs the repository's own
`ml/train_span_extractor.py`, so the split, the seed and the evaluation are
the same code as a local run. A copy of that logic here could drift from it.

What the bundle contains: the 2,466 slice-verified tagged segments derived
from OPP-115, plus the four modules that build, split and train on them. The
OPP-115 corpus itself is not uploaded.

**Runtime → Change runtime type → GPU** before running.

## 1. Install

In [ ]:
!pip -q install "transformers==4.46.3" "torch" "accelerate" 2>&1 | tail -2
import torch
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU. Runtime -> Change runtime type -> GPU, then rerun.")

## 2. Upload the bundle

Run `python ml/export_f12_bundle.py` locally first; it writes
`ml/runs/f12_training_bundle.zip` (about 1.7 MB). Upload that file here.

In [ ]:
from google.colab import files           # On Kaggle: use the file panel instead.
uploaded = files.upload()                # choose f12_training_bundle.zip

import zipfile
name = next(iter(uploaded))
with zipfile.ZipFile(name) as archive:
    archive.extractall("project")
    print("\n".join(archive.namelist()))

## 3. Verify the dataset before training

The trainer re-checks that every token slices out of its segment and refuses
to train if any does not. A `--dry-run` shows the split and the per-label
support without touching the GPU, so a damaged upload is caught in seconds
rather than after an epoch.

In [ ]:
%cd project
!python ml/train_span_extractor.py --dry-run

## 4. Train

Four epochs of `distilbert-base-uncased` over 1,713 training segments. On a
free T4 this is a few minutes.

The test split is read **once**, at the end. Validation scores are printed
each epoch and are what any epoch or threshold choice should be made against.

Scores are span-level and exact: a predicted span counts only when its type,
its value and both its offsets match a gold span. Overlapping scores are
printed beside them so the boundary error rate is visible rather than hidden
behind whichever definition flatters more.

In [ ]:
!python ml/train_span_extractor.py --epochs 4 --batch-size 16

## 5. Download the checkpoint

Unzip into `ml/checkpoints/span-extractor-v1/` in the repository. The backend
looks for it there and refuses to run F-12 if it is absent — it never
substitutes a heuristic tagger, because a different tagger would move span
boundaries and the same policy would then produce different evidence.

Copy the printed test table into `docs/M10_REPORT.md`. Do not record any
figure this notebook did not actually print.

In [ ]:
!cd ml/checkpoints && zip -qr /content/span-extractor-v1.zip span-extractor-v1
!ls -la /content/span-extractor-v1.zip
from google.colab import files
files.download("/content/span-extractor-v1.zip")

In [ ]:
# The metrics file, for the record. Paste this into the M10 report too.
!cat ml/runs/span-extractor-v1-metrics.json